# Libs

In [ ]:
import pandas as pd
import numpy as np
from numpy import linalg as LA
import re

import matplotlib.pyplot as plt
import seaborn as sns

from scipy.sparse import csgraph

from wordcloud import WordCloud
import math

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans, SpectralClustering
from sklearn.metrics import silhouette_score, calinski_harabasz_score
from sklearn.manifold import TSNE
from sklearn.preprocessing import MultiLabelBinarizer, Normalizer
from sklearn.neighbors import kneighbors_graph

# 1. Pré-processamento dos dados

In [ ]:
## Lendo e exibindo o dataset
df_dataset = pd.read_csv('steam_games.csv', decimal='.')
df_dataset

- Como os jogos podem pertencer a mais de um gênero e nesses casos estão separados por ";" converti para listas com o objetivo de facilitar a manipulação

In [ ]:
## Removendo colunas desnecessárias
df_dataset.drop(columns=['appid', 'release_date', 'positive_ratings', 'negative_ratings', 'average_playtime', 'price'], inplace=True, errors='ignore')
## Convertendo a coluna de gêneros em listas
df_dataset['genres'] = df_dataset['genres'].str.split(';')
df_dataset

In [ ]:
## Analisando a distribuição dos gêneros
count_generos = df_dataset['genres'].explode().value_counts()
count_generos_percentual = df_dataset['genres'].explode().value_counts(normalize=True) * 100
df_generos = pd.DataFrame({
    'Gênero': count_generos.index,
    'Absoluto': count_generos.values,
    'Percentual (%)': count_generos_percentual.values
})
df_generos['Percentual (%)'] = df_generos['Percentual (%)'].round(2)
df_generos

In [ ]:
## Plotando a matriz de co-ocorrência de gêneros
mlb = MultiLabelBinarizer()
genre_matrix = pd.DataFrame(mlb.fit_transform(df_dataset['genres']), columns=mlb.classes_)
coocc_matrix = genre_matrix.T.dot(genre_matrix)

mask = np.triu(np.ones_like(coocc_matrix, dtype=bool), k=0)
plt.figure(figsize=(25, 20)) 
sns.heatmap(
    coocc_matrix, 
    mask=mask, 
    annot=True, 
    fmt='d',
    cmap='Reds', 
    cbar=False,
    square=True,
    annot_kws={"size": 12} 
)

plt.xticks(fontsize=10, rotation=90)
plt.yticks(fontsize=10, rotation=0)

plt.title('Matriz Co-ocorrência de Gêneros', fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
## Removendo gêneros pouco representativos
LIMITE_OCORRENCIA= 800  # Gêneros com menos de 800 ocorrências serão removidos

contagem_total = df_dataset['genres'].explode().value_counts()

generos_raros = set(contagem_total[contagem_total < LIMITE_OCORRENCIA].index)
print(f"Total de gêneros únicos marcados como pouco representativos: {len(generos_raros)}")

def limpar_generos_raros(lista_generos):
    return [g for g in lista_generos if g not in generos_raros]

nao_generos = {'Free to Play', 'Early Access'}
def limpar_nao_generos(lista_generos):
    return [g for g in lista_generos if g not in nao_generos]

df_dataset['genres'] = df_dataset['genres'].apply(limpar_generos_raros)
df_dataset['genres'] = df_dataset['genres'].apply(limpar_nao_generos)

linhas_antes = len(df_dataset)
df_dataset = df_dataset[df_dataset['genres'].map(len) > 0].copy()
linhas_depois = len(df_dataset)

df_dataset.reset_index(drop=True, inplace=True)

print(f"Linhas removidas (jogos sem gêneros representativos): {linhas_antes - linhas_depois}")
print(f"Novo tamanho do dataset: {linhas_depois}")

df_dataset

In [ ]:
## Analisando a distribuição dos gêneros após limpeza
count_generos = df_dataset['genres'].explode().value_counts()
count_generos_percentual = df_dataset['genres'].explode().value_counts(normalize=True) * 100
df_generos = pd.DataFrame({
    'Gênero': count_generos.index,
    'Absoluto': count_generos.values,
    'Percentual (%)': count_generos_percentual.values
})
df_generos['Percentual (%)'] = df_generos['Percentual (%)'].round(2)
df_generos

- O título do jogo carrega palavras-chave que podem ajudar a definir o gênero do jogo, por isso optei por incluir também o título na análise. 
- Percebi que a descrição curta já carrega muitas informações relevantes e assim inialmente optei por utilizar ela com objetivo de reduzir ruído para o TF-IDF.

In [ ]:
## Removendo caracteres especiais, stopwords e aplicando lematização
stop_words = set(stopwords.words('english'))
domain_stopwords = {
    # Termos genéricos de Jogos
    'game', 'play', 'player', 'steam', 'feature', 'experience', 'world', 'new', 'best', 'use', 'way','go',
    
    # Termos de comercialização dos jogos
    'free', 'edition', 'version', 'available', 'classic', 'original', 'pack', 'dlc', 
    'soundtrack', 'access', 'early', 'development', 'community', 'support', 'content', 
    'include', 'includes', 'great', 'unique', 'award', 'winning', 'fun', 'enjoy', 'real',
    
    # Termos mais técnicos e de plataforma
    'mode', 'system', 'level', 'control', 'controller', 'graphics', 'visual', 'audio', 
    'environment', 'mechanic', 'design', 'engine', 'screen', 'keyboard', 'mouse', 'pc',
    
    # Termos genéricos para vários gêneros
    'character', 'story', 'life', 'way', 'time', 'make', 'take', 'get', 'help', 'find', 
    'challenge', 'skill', 'different', 'various', 'many', 'set', 'place', 'like', 'one', 'two'
}
stop_words.update(domain_stopwords)

lemmatizer = WordNetLemmatizer()

def clean_text(text):
# Remove caracteres especiais e números (mantém apenas letras)
    text = re.sub(r'[^a-zA-Z\s]', '', str(text).lower())
    tokens = []
    for w in text.split():
        if w not in stop_words:
            lemma = lemmatizer.lemmatize(w, pos='v') 
            if lemma == w:
                lemma = lemmatizer.lemmatize(w)
            tokens.append(lemma)
    return ' '.join(tokens)

df_dataset['clean_text'] = df_dataset['short_description'].apply(clean_text)
df_dataset

# 2. TF-IDF

In [ ]:
tfidf = TfidfVectorizer(
    min_df=10,
    max_df=0.85,
    ngram_range=(1, 2),
    stop_words=list(stop_words),
    norm='l2'
)
X_tfidf = tfidf.fit_transform(df_dataset['clean_text'])
X_tfidf

# 3. Redução de dimensionalidade

 - Utilizar o PCA na matrix sparsa gerada é muito caro, por isso foi utilizado o TruncatedSVD que consegue trabalhar com matrizes esparsas
 - O número de componentes foi definido de modo a reter aproximadamente 20% da variância para que menos ruído fosse mantido

In [ ]:
svd_analysis = TruncatedSVD(n_components=1000, random_state=42)
svd_analysis.fit(X_tfidf)

variance_cumsum = np.cumsum(svd_analysis.explained_variance_ratio_)

plt.figure(figsize=(10, 6))
plt.plot(range(1, 1001), variance_cumsum, linewidth=2)
plt.title('Variância Acumulada por Componente (SVD)')
plt.xlabel('Número de Componentes')
plt.ylabel('Variância Explicada Acumulada')
plt.grid(True)
plt.show()

- Para dados textuais manter uma alta variância implica em manter muito ruído nos dados (palavras que não diferenciam bem os gêneros), por isso foi escolhido um número de componentes igual a 200 que mantém ~20% da variância dos dados.

In [ ]:
svd = TruncatedSVD(n_components=200, random_state=42) 
X_svd = svd.fit_transform(X_tfidf)

normalizer = Normalizer(norm='l2')
X_svd = normalizer.transform(X_svd) 

print(f"Variância explicada: {svd.explained_variance_ratio_.sum():.2f}")
print(f"Shape final: {X_svd.shape}")

# 4. Clustering

## Visualização em 2D utilizando t-SNE

### t-SNE

In [ ]:
tsne = TSNE(
    n_components=2,
    perplexity=50,
    early_exaggeration=12.0,
    learning_rate='auto',
    max_iter=1000,
    n_iter_without_progress=500,
    min_grad_norm=1e-7,
    metric='euclidean',
    init='pca',
    random_state=42,
    method='barnes_hut',
    angle=0.5,
    verbose=0,
    n_jobs=None
)

X_embedded = tsne.fit_transform(X_svd)
plt.figure(figsize=(8, 6))
plt.scatter(X_embedded[:, 0], X_embedded[:, 1], s=40, alpha=0.7, c='red')
plt.title("Visualização com t-SNE dos dados reduzidos por SVD")
plt.xlabel("Component 1")
plt.ylabel("Component 2")
plt.show()

## KMeans

In [ ]:
## Aplicando busca para o melhor K usando método do cotovelo, silhouette score e calinski-harabasz
unique_genres = df_dataset['genres'].explode().unique()
K_range = range(2, len(unique_genres) + 15)

inertia = []
silhouette_scores = []
calinski_scores = []

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_svd)
    
    inertia.append(kmeans.inertia_)
    
    silhouette_avg = silhouette_score(X_svd, labels)
    silhouette_scores.append(silhouette_avg)
    
    ch_score = calinski_harabasz_score(X_svd, labels)
    calinski_scores.append(ch_score)

plt.figure(figsize=(18, 5))

# Gráfico 1: Cotovelo (Inércia)
plt.subplot(1, 3, 1)
plt.plot(K_range, inertia, marker='o')
plt.title('Método do Cotovelo (Inércia)')
plt.xlabel('K')
plt.ylabel('Inércia')
plt.grid(True)

# Gráfico 2: Silhouette Score
plt.subplot(1, 3, 2)
plt.plot(K_range, silhouette_scores, marker='o', color='green')
plt.title('Silhouette Score')
plt.xlabel('K')
plt.ylabel('Score')
plt.grid(True)

# Gráfico 3: Calinski-Harabasz
plt.subplot(1, 3, 3)
plt.plot(K_range, calinski_scores, marker='o', color='orange')
plt.title('Calinski-Harabasz Score')
plt.xlabel('K')
plt.ylabel('Score')
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Definindo K baseado no número de gêneros únicos observados
K = 10

# Algoritmo A: K-Means
kmeans = KMeans(n_clusters=K, random_state=42, n_init=10)
labels_kmeans = kmeans.fit_predict(X_svd)

In [ ]:
df_dataset['cluster'] = labels_kmeans
df_exploded = df_dataset.explode('genres')

cross_tab_pct = pd.crosstab(df_exploded['cluster'], df_exploded['genres'], normalize='index')
cross_tab_pct = cross_tab_pct * 100

def get_top_genres_per_cluster(cross_tab_pct, top_n=5):
    print(f"{'Cluster':<10} | {'Principais Gêneros (com %)'}")
    print("-" * 60)
    
    for cluster_id in cross_tab_pct.index:
        top_genres = cross_tab_pct.loc[cluster_id].nlargest(top_n)
        genre_str = ", ".join([f"{g} ({v:.1f}%)" for g, v in top_genres.items()])
        print(f"{cluster_id:<10} | {genre_str}")

get_top_genres_per_cluster(cross_tab_pct)

In [ ]:
clusters_unicos = sorted(df_dataset['cluster'].unique())
n_clusters = len(clusters_unicos)
n_cols = 3
n_rows = math.ceil(n_clusters / n_cols)

plt.figure(figsize=(20, 5 * n_rows))

for i, cluster_id in enumerate(clusters_unicos):
    ax = plt.subplot(n_rows, n_cols, i + 1)
    
    text_cluster = " ".join(df_dataset[df_dataset['cluster'] == cluster_id]['clean_text'].astype(str))
    
    wordcloud = WordCloud(
        width=800, 
        height=400,
        background_color='white',
        stopwords=stop_words,
        max_words=50,
        colormap='viridis'
    ).generate(text_cluster)
    
    ax.imshow(wordcloud, interpolation='bilinear')
    ax.set_title(f"Cluster {cluster_id}", fontsize=16)
    ax.axis('off')

plt.tight_layout()
plt.show()

## Spectral Clustering

In [ ]:
# Construindo a matriz de adjacências do grafo de vizinhos mais próximos.
G = kneighbors_graph(X_svd, n_neighbors = 10, include_self = True)
A = 0.5 * (G + G.T)

# Construindo a Laplaciana Normalizada
L = csgraph.laplacian(A, normed = True).todense()

# Obtendo os autovalores da Laplaciana Normalizada
values, _ = LA.eigh(L)

# Plotando os valores dos 'gaps' e escolhendo um k adequado.
plt.scatter([i for i in range(2, len(unique_genres) + 15)], values[:20])
plt.xlabel('Índice do autovalor')
plt.ylabel('Autovalor');

In [ ]:
## Verificando maior gap entre os autovalores
gaps = np.diff(values)
K_spectral = np.argmax(gaps[:20]) + 1
print(f"O maior gap entre autovalores ocorre no índice: {K_spectral}")

In [ ]:
spectral = SpectralClustering(n_clusters = 10, affinity = 'nearest_neighbors', n_neighbors = 10)
labels_spectral = spectral.fit_predict(X_svd)

In [ ]:
tsne = TSNE(
    n_components=2,
    perplexity=50,
    early_exaggeration=12.0,
    learning_rate='auto',
    max_iter=1000,
    n_iter_without_progress=500,
    min_grad_norm=1e-7,
    metric='euclidean',
    init='pca',
    random_state=42,
    method='barnes_hut',
    angle=0.5,
    verbose=0,
    n_jobs=None
)

X_embedded = tsne.fit_transform(X_svd)
plt.figure(figsize=(10, 8))
sc = plt.scatter(
    X_embedded[:, 0], 
    X_embedded[:, 1], 
    s=40, 
    alpha=0.7, 
    c=labels_spectral,
    cmap='tab20'
)

plt.colorbar(sc, label='Cluster ID')
plt.title("Visualização com t-SNE dos clusters - Spectral Clustering")
plt.xlabel("Component 1")
plt.ylabel("Component 2")
plt.show()

In [ ]:
df_dataset['cluster_spectral'] = labels_spectral
df_exploded = df_dataset.explode('genres')

cross_tab_pct = pd.crosstab(df_exploded['cluster_spectral'], df_exploded['genres'], normalize='index')
cross_tab_pct = cross_tab_pct * 100

def get_top_genres_per_cluster(cross_tab_pct, top_n=5):
    print(f"{'Cluster':<10} | {'Principais Gêneros (com %)'}")
    print("-" * 60)
    
    for cluster_id in cross_tab_pct.index:
        top_genres = cross_tab_pct.loc[cluster_id].nlargest(top_n)
        genre_str = ", ".join([f"{g} ({v:.1f}%)" for g, v in top_genres.items()])
        print(f"{cluster_id:<10} | {genre_str}")

get_top_genres_per_cluster(cross_tab_pct)

In [ ]:
clusters_unicos = sorted(df_dataset['cluster_spectral'].unique())
n_clusters = len(clusters_unicos)
n_cols = 3
n_rows = math.ceil(n_clusters / n_cols)

plt.figure(figsize=(20, 5 * n_rows))

for i, cluster_id in enumerate(clusters_unicos):
    ax = plt.subplot(n_rows, n_cols, i + 1)
    
    text_cluster = " ".join(df_dataset[df_dataset['cluster_spectral'] == cluster_id]['clean_text'].astype(str))
    
    wordcloud = WordCloud(
        width=800, 
        height=400,
        background_color='white',
        stopwords=stopwords,
        max_words=50,
        colormap='viridis'
    ).generate(text_cluster)
    
    ax.imshow(wordcloud, interpolation='bilinear')
    ax.set_title(f"Cluster {cluster_id}", fontsize=16)
    ax.axis('off')

plt.tight_layout()
plt.show()